# Notebook 03 — Loss Surface Geometry & Optimization Dynamics

**Session:** Linear Regression — L1, L2, and Huber Losses (Level 1)  
**Scheduled Date:** 2026-11-23 · **Module:** 02 — Calculus and Linear Algebra

---

## 1. Overview and Objectives

Understanding optimization requires visualizing the geometry of the objective function:
1. The parameter space $(w, b)$ defines a two-dimensional loss landscape $\mathcal{L}(w, b)$.
2. For linear regression with squared error, this surface is an elliptic paraboloid with a unique global minimum $(w^*, b^*)$.
3. We will:
   - Compute the exact closed-form analytical solution.
   - Map 2D contour lines and the 3D loss surface.
   - Trace gradient descent trajectories under varying learning rates $\eta$.
   - Implement principled stopping criteria based on loss tolerances and gradient norms.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 2. Closed-Form Ordinary Least Squares (OLS) Solution

For univariate regression on dataset $\{(x_i, y_i)\}_{i=1}^N$, setting $\nabla \mathcal{L} = \mathbf{0}$ yields:
$$w^* = \frac{\sum_{i=1}^N (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^N (x_i - \bar{x})^2} = \frac{\text{Cov}(x, y)}{\text{Var}(x)}$$
$$b^* = \bar{y} - w^* \bar{x}$$


In [ ]:
data = np.loadtxt(Path("../data/house_prices_toy.csv"), delimiter=",", skiprows=1)
x = data[:, 0]
y = data[:, 1]
N = len(x)

x_bar = np.mean(x)
y_bar = np.mean(y)

w_star = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
b_star = y_bar - w_star * x_bar

print(f"Dataset points: N={N}")
print(f"Feature mean:  x_bar={x_bar:.4f}")
print(f"Label mean:    y_bar={y_bar:.4f}")
print(f"Analytical optimal parameters: w*={w_star:.4f}, b*={b_star:.4f}")

## 3. Mapping the 2D Loss Surface

We evaluate the Mean Squared Error over a dense grid of candidate parameter values $(w, b)$.


In [ ]:
w_grid = np.linspace(-1.0, 3.5, 120)
b_grid = np.linspace(-2.0, 4.5, 120)
W, B = np.meshgrid(w_grid, b_grid)

# Vectorized MSE loss computation
# y_hat shape: (120, 120, N)
Y_hat = W[:, :, None] * x[None, None, :] + B[:, :, None]
Loss_grid = np.mean((Y_hat - y[None, None, :]) ** 2, axis=-1)

# Plot contour map
plt.figure(figsize=(7.5, 5))
contours = plt.contour(
    W, B, Loss_grid, levels=np.logspace(np.log10(np.min(Loss_grid) + 0.01), 2.5, 25), cmap="viridis"
)
plt.clabel(contours, inline=True, fontsize=8, fmt="%.1f")
plt.scatter(
    [w_star],
    [b_star],
    color="red",
    marker="*",
    s=150,
    zorder=5,
    label=f"Optimal (w*={w_star:.2f}, b*={b_star:.2f})",
)

plt.title("MSE Loss Contours $\\mathcal{L}(w, b)$ over Parameter Space")
plt.xlabel("Weight w")
plt.ylabel("Bias b")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Optimization Trajectories Under Varying Learning Rates

We simulate gradient descent starting from initial point $(w_0 = -0.34, b_0 = 0.04)$ with three
distinct learning rates:
1. $\eta = 0.005$ (Sub-optimal, slow convergence)
2. $\eta = 0.02$ (Stable, rapid convergence)
3. $\eta = 0.065$ (Oscillatory near stability threshold)


In [ ]:
def run_gradient_descent(
    x: np.ndarray, y: np.ndarray, w0: float, b0: float, lr: float, steps: int = 60
) -> tuple[np.ndarray, np.ndarray, list[float]]:
    w, b = w0, b0
    n = len(x)
    traj_w = [w]
    traj_b = [b]
    loss_hist = []

    for _ in range(steps):
        r = (w * x + b) - y
        loss = float(np.mean(r**2))
        loss_hist.append(loss)
        dw = (2.0 / n) * np.sum(x * r)
        db = (2.0 / n) * np.sum(r)
        w -= lr * dw
        b -= lr * db
        traj_w.append(w)
        traj_b.append(b)
    return np.array(traj_w), np.array(traj_b), loss_hist


w_init, b_init = -0.34, 0.04
lr_configs = [
    (0.005, "tab:orange", "η = 0.005 (Slow)"),
    (0.02, "tab:green", "η = 0.020 (Optimal)"),
    (0.065, "tab:purple", "η = 0.065 (Oscillatory)"),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Contour plot with trajectories
ax1.contour(W, B, Loss_grid, levels=25, cmap="Blues_r", alpha=0.6)
ax1.scatter([w_star], [b_star], color="red", marker="*", s=160, zorder=6, label="Minimum (w*, b*)")

for lr, col, lbl in lr_configs:
    tw, tb, lh = run_gradient_descent(x, y, w_init, b_init, lr=lr, steps=40)
    ax1.plot(tw, tb, marker="o", markersize=3.5, color=col, lw=1.5, label=lbl, alpha=0.85)
    ax2.plot(lh, color=col, lw=1.8, label=lbl)

ax1.set_title("Optimization Trajectories on Loss Surface")
ax1.set_xlabel("Weight w")
ax1.set_ylabel("Bias b")
ax1.legend(loc="upper left", fontsize=9)

ax2.set_title("Convergence Speed across Learning Rates")
ax2.set_xlabel("Iteration")
ax2.set_ylabel("MSE Loss")
ax2.set_yscale("log")
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Formal Stopping Criteria

In practical machine learning systems, gradient descent loops run until satisfying one of three termination predicates:
1. **Gradient Norm Tolerance:** $\|\nabla \mathcal{L}\|_2 = \sqrt{(\partial \mathcal{L}/\partial w)^2 + (\partial \mathcal{L}/\partial b)^2} < \epsilon_{\text{grad}}$
2. **Loss Improvement Tolerance:** $|\mathcal{L}^{(t)} - \mathcal{L}^{(t-1)}| < \epsilon_{\text{loss}}$
3. **Maximum Epochs Safeguard:** $t \ge t_{\max}$


In [ ]:
def train_with_stopping_criteria(
    x: np.ndarray,
    y: np.ndarray,
    lr: float = 0.02,
    tol_grad: float = 1e-4,
    tol_loss: float = 1e-6,
    max_epochs: int = 1000,
) -> dict:
    w, b = -0.34, 0.04
    n = len(x)
    prev_loss = float("inf")

    for epoch in range(1, max_epochs + 1):
        r = (w * x + b) - y
        loss = float(np.mean(r**2))
        dw = (2.0 / n) * np.sum(x * r)
        db = (2.0 / n) * np.sum(r)
        grad_norm = float(np.sqrt(dw**2 + db**2))

        # Check stopping criteria
        if grad_norm < tol_grad:
            return {
                "status": "converged_grad_norm",
                "epochs": epoch,
                "w": w,
                "b": b,
                "loss": loss,
                "grad_norm": grad_norm,
            }
        if abs(prev_loss - loss) < tol_loss:
            return {
                "status": "converged_loss_delta",
                "epochs": epoch,
                "w": w,
                "b": b,
                "loss": loss,
                "grad_norm": grad_norm,
            }

        prev_loss = loss
        w -= lr * dw
        b -= lr * db

    return {
        "status": "max_epochs_reached",
        "epochs": max_epochs,
        "w": w,
        "b": b,
        "loss": loss,
        "grad_norm": grad_norm,
    }


res = train_with_stopping_criteria(x, y, lr=0.02)
print("Stopping Criteria Simulation Results:")
for k, v in res.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

assert np.isclose(res["w"], w_star, atol=1e-2)
assert np.isclose(res["b"], b_star, atol=1e-2)

## 6. Summary and Transition to Level 2

- The loss surface of univariate linear regression is strictly convex with a unique global minimum.
- The analytical solution $(w^*, b^*)$ matches the asymptotic convergence point of gradient descent.
- Learning rate $\eta$ governs optimization dynamics: too small leads to sluggish convergence; too large triggers oscillation or catastrophic divergence.
- In **Level 2**, we generalize these concepts to multi-variable inputs (vectorization with matrices), feature scaling, and stochastic vs. mini-batch versus full-batch regimes.
